# 19 — Word2Vec

Word2Vec learns dense, low-dimensional vectors where words that appear in similar contexts end up close together: "python" sits near "nlp" and "tensorflow" because they co-occur in the same sentences. Unlike BoW (Ch. 15) it captures *meaning*, and unlike TF-IDF (Ch. 16) the vectors are learned, not counted.

**Why it matters for resumes / ATS:** exact-keyword matching fails on synonyms ("pytorch" vs "torch", "ML" vs "machine learning"). Embeddings let a matcher say "this resume mentions things semantically close to what the JD asks for" — the difference between a keyword grep and a semantic search.

**Goal:** Learn dense vector representations that capture semantic meaning.

Using gensim on a tiny 6-sentence toy corpus, this chapter trains a model, inspects the vectors, and probes them with similarity, nearest-neighbor, and document-level queries. The corpus is small on purpose: the *mechanics* are what matter here; scale comes in Ch. 21–22 with pretrained models.

## 1. Training a Small Word2Vec Model

`Word2Vec(sentences, vector_size=50, window=3, min_count=1, epochs=100)` trains by predicting each word from its neighbors (CBOW) or neighbors from the word (skip-gram). `window=3` sets the context radius, `min_count=1` keeps every token in this toy corpus, and 100 epochs let the tiny dataset converge.

**What the code does:** trains on six hand-written sentences and reports the vocabulary and a sample vector.
- Vocabulary size: `26` unique tokens.
- `model.wv['python']` has shape `(50,)`; its first 10 dims are small floats like `[-0.016, 0.009, -0.008, ...]` — dense and distributed, nothing like the 0/1 BoW rows.

**Try it:** each dimension means something only in combination — the "meaning" is spread across all 50 floats. Values vary run-to-run because training starts from a random init.

In [ ]:
from gensim.models import Word2Vec
sentences = [
    ["python", "is", "great", "for", "nlp", "and", "machine", "learning"],
    ["tensorflow", "is", "a", "deep", "learning", "framework"],
    ["data", "scientist", "uses", "python", "for", "analysis"],
    ["machine", "learning", "engineer", "builds", "models"],
    ["nlp", "engineer", "works", "with", "text", "data"],
    ["python", "developer", "writes", "code", "daily"],
]
model = Word2Vec(sentences, vector_size=50, window=3, min_count=1, epochs=100)
print(f"Vocabulary size: {len(model.wv)}")
print(f"Vector for 'python': shape {model.wv['python'].shape}")
print(f"First 10 dims: {model.wv['python'][:10].round(3)}")

## 2. Word Similarity

`model.wv.similarity(w1, w2)` returns the cosine of the two learned vectors. In a well-trained model, semantically related words score higher than unrelated ones.

**What the code does:** scores "python" against five partners from the toy corpus.
- This run: `sim('python', 'great') = 0.235`, `sim('python', 'code') = 0.018`, `sim('python', 'data') = -0.170` — values shift between runs, but the relative ordering reflects the tiny corpus's co-occurrence structure.

**Try it:** with only 6 sentences, similarities are noisy — "python" appears with "nlp" and "machine learning" but also with "great" and "code". Don't over-read exact numbers; the API and the direction of the signal are the lesson.

In [ ]:
pairs = [("python", "nlp"), ("python", "tensorflow"), ("python", "data"),
         ("python", "code"), ("python", "great")]
for w1, w2 in pairs:
    if w1 in model.wv and w2 in model.wv:
        sim = model.wv.similarity(w1, w2)
        print(f"  sim('{w1}', '{w2}') = {sim:.3f}")

## 3. Most Similar Words

`most_similar(word, topn=k)` ranks the whole vocabulary by cosine to the query and returns the k nearest neighbors — the model's own guess at what "goes with" a word.

**What the code does:** queries three words and prints their top-3 neighbors.
- This run: near `'python'`: `great (0.235)`, `builds (0.229)`, `scientist (0.161)`; near `'learning'`: `uses (0.279)`, `works (0.225)`, `tensorflow (0.198)`.

**Try it:** the neighbors of "learning" lean technical (`tensorflow`) while "python" picks up both technical and generic words — the corpus's co-occurrence structure showing through. Exact numbers vary per run.

In [ ]:
for word in ["python", "learning", "engineer"]:
    if word in model.wv:
        similar = model.wv.most_similar(word, topn=3)
        print(f"\nWords similar to '{word}':")
        for w, s in similar:
            print(f"  {w:12s} {s:.3f}")

## 4. Word Analogies

The classic embedding demo is vector arithmetic: `king - man + woman ≈ queen`. Here the notebook applies the same `most_similar(positive=[...])` API in a simpler form — "what's closest to 'python'?" — because a 6-sentence toy corpus is too small for reliable analogies.

**What the code does:** wraps `most_similar(positive=["python"], topn=3)` in a `try/except KeyError` guard for words missing from the vocabulary.
- This run: `[('great', 0.235), ('builds', 0.229), ('scientist', 0.161)]` — the same neighbors as the plain similarity query, since `positive=[...]` is exactly that query.

**Try it:** on a pretrained model (Ch. 21) try `most_similar(positive=["queen"], negative=["king"], topn=3)`-style arithmetic; here the point is the API shape and the `KeyError` guard on out-of-vocabulary words.

In [ ]:
# Classic: king - man + woman = queen
try:
    result = model.wv.most_similar(positive=["python"], topn=3)
    print(f"Similar to 'python': {result}")
except KeyError as e:
    print(f"Word not in vocabulary: {e}")

## 5. Averaging Word Vectors for Document Similarity

To turn word vectors into a document vector, average the word vectors: `doc_vector()` filters to known words and takes the mean. It's crude but effective — the standard baseline before sentence transformers (Ch. 22).

**What the code does:** builds vectors for three docs and computes pairwise cosine similarity.
- Doc1 vs Doc2 (ML-related): this run ≈ `0.14`; Doc1 vs Doc3 (Java/backend): ≈ `0.14` — nearly tied, because the toy model's vectors are noisy and doc 3's words (`java`, `spring`, `backend`) are mostly out of vocabulary, so its vector is built from almost nothing.

**Try it (known issue):** this cell calls `cosine_similarity`, but that import lives in Ch. 18's notebook, not here — add `from sklearn.metrics.pairwise import cosine_similarity` to run it. It also silently drops OOV words, which is why doc 3's "vector" is mostly the single known word `developer`.

In [ ]:
def doc_vector(words, model):
    words = [w for w in words if w in model.wv]
    if not words:
        return None
    return np.mean([model.wv[w] for w in words], axis=0)

import numpy as np
docs = [
    "python machine learning nlp",
    "tensorflow deep learning framework",
    "java spring backend developer",
]
vectors = [doc_vector(d.split(), model) for d in docs]
if vectors[0] is not None and vectors[1] is not None:
    sim_ml = cosine_similarity([vectors[0]], [vectors[1]])[0][0]
    sim_diff = cosine_similarity([vectors[0]], [vectors[2]])[0][0]
    print(f"\nDocument similarity:")
    print(f"  Doc1 vs Doc2 (ML related):     {sim_ml:.3f}")
    print(f"  Doc1 vs Doc3 (different area): {sim_diff:.3f}")

## Key Insight: Word2Vec captures semantics — 'python' is close to 'nlp' AND 'tensorflow'.

**Dense embeddings turn "related in text" into "close in vector space" — the first semantic representation in this series.**

Even on six sentences, the mechanics are clear: context-based training, cosine probes, nearest neighbors, vector arithmetic, and document-level averaging. The limits (tiny vocabulary, no OOV handling, noisy similarities) are exactly what Ch. 20's FastText subwords and Ch. 21–22's pretrained transformers fix.